In [1]:
from functions.get_all_geometries import *

In [2]:
from functions.sparql_requests import sparql_select, sparql_update

# 1. Gather data & build the geographic knowledge graph

## 1.1 Get NUTS rdf data
The NUTS rdf data is provided [on the eurostat website](https://op.europa.eu/en/web/eu-vocabularies/dataset/-/resource?uri=http://publications.europa.eu/resource/dataset/nuts). The latest version is from 2024. Here some NUTS areas from the 2021 version are deprecated. This includes all NUTS regions in the United Kingdom due to BREXIT but also some NUTS level 2 regions in Portugal or some level 3 regions all around Europe. These are probably due to updates in these regions (e.g. mergers or seperations of regions).

The graph does not provide geodata directly. Only a link to a geojson file [(example)](https://gisco-services.ec.europa.eu/distribution/v2/nuts/distribution/FRK27-region-20m-3857-2021.geojson). These provided geometries however seem to be the ones from 2021 (also evident due to the "2021" in the link. Newer NUTS regions (e.g. PT19, level 2) that do only exist since the 2024 version, do not have a geometry defined yet.

Due to these contrains, the geometries from 2021 will be used. This also aligns with the fact, that some LLMs used in this study will have a cutoff-date before the 2024 release.

The original concept model for the NUTS-rdf data is pretty high level and uses well known namespaces such as skos:, dcat:, and locn:

However, in order to perform spatial analysis with the NUTS data, a GeoSPARQL compliant format is needed. Therefore the ontology is altered to meet these standards.

![pic](data_external/pictures/original_ontology.png)

In [3]:
# Define new Ontology that leverages GEOSPARQL concepts

The original ontology uses locn:geometry 1,..n to link the geodata to a NUTS-code. This entails multiple geometries per NUTS-code (depending on the different reference-systems and scales that are provided.

For simplicity reasons this ontology will only use one single geometry per NUTS-code. The geometry is defined by `?nutscode nutsdef:scale "10m"` and `?nutscode nutsdef:projection "WGS84"`. The comperativly high scale of of 10m is used since the sutdy manly focuses on topological relations, which are still well enough represented with this scale.

## 1.2 Set up libraries, endpoints, the repository and superfunctions

In [4]:
import io
import requests
import pandas as pd

In [5]:
graphdb_server_url = 'http://localhost:7200'
repository_id = 'test_geosparql'
select_endpoint_url = "http://localhost:7200/repositories/test_geosparql"
update_endpoint_url = "http://localhost:7200/repositories/test_geosparql/statements"
base_iri_geometry = "http://geonuts.eu/geometry/"

# namespace = "http://geonuts.eu/"

### 1.2. Set up repository
The repository is set up in GraphDB with standard settings. It is important that the GeoSPARQL extention is enabled.

<img src="pics/create_repo.jpg" alt="create_repo" width="600"/>

## 1.3 Build GeoSPARQL compliant graph structure

### 1.3.1 Set rdf:type to geo:Feature
In order for the GeoSPARQL extention to work, objects with the relation `?object geo:hasGeometry geo:Geometry` has to be `?object rdf:type geo:Feature` since we attach the `geo:Geometry` to our NUTS codes objects, we want to add the type `geo:Feature` to every object where `?object rdf:type skos:Concept`.

In [6]:
with open("sparql/set_rdfType_geoFeature.sparql", "r") as f:
    query_setRdfTypeGeoGeometry = f.read()

Execute the query

In [7]:
# sparql_update(query_setRdfTypeGeoGeometry)

### 1.3.x Remove all placeholder objects
In for each NUTS-level in every country there are placeholder NUTS-codes. These are defined after the schema CONTRYCODEZZZ (e.g. ATZ, DEZZ, or LUZZZ, depending on the NUTS level). Later, these should be removed as they compromise queries and results later.

The used query selects the respective NUTS code objects (skos:Concept) via their relation skos:prefLable that contains "Extra" in the literal. Then all relations from this NUTS code are removed.

In [8]:
query_removeZZZ = """
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

DELETE {
    ?code ?rel ?o.
}
WHERE {
    ?code ?rel ?o .
    ?code skos:prefLabel ?label .
    FILTER CONTAINS(STR(?label), "Extra")
}
"""

_ = sparql_update(query_removeZZZ, update_endpoint_url)

SUCESSFUL UPDATE


### 1.3.2 Get all geometries from the graph
The query `query_getAllGeoms` takes all the geometries download links from the graph.
`get_geojson` then sends a request to the respective download links and the returned WKT geometries are saved in the column `geometry` of the dataframe.

In [9]:
with open("sparql/get_all_geometry_links_10m_WGS84_also_deprecated.sparql", "r") as f:
    query_getAllGeoms = f.read()

# get a download link for each geometry
df = sparql_select(query_getAllGeoms, select_endpoint_url)

if not len(df.nutscode) == df.nutscode.nunique():
    raise AssertionError("There should only be one retrieved geometry for each NUTS-code!")

In [11]:
df = df.sample(10)
df['geometry'] = df.apply(get_geojson, axis=1)

df.head()

,geometry_index,nutscode,link,scale,projection,geometry
991,http://data.europa.eu/nuts/geometry/2byqdozz,FR108,https://gisco-services.ec.europa.eu/distributi...,10m,WGS84,"POLYGON ((2.59053 49.07965, 2.55306 49.00982, ..."
401,http://data.europa.eu/nuts/geometry/wmumkf60,UKC11,https://gisco-services.ec.europa.eu/distributi...,10m,WGS84,"POLYGON ((-1.26125 54.5719, -1.27078 54.52493,..."
329,http://data.europa.eu/nuts/geometry/zfwm1kwr,SI035,https://gisco-services.ec.europa.eu/distributi...,10m,WGS84,"POLYGON ((15.1549 46.09341, 15.08114 46.07352,..."
17,http://data.europa.eu/nuts/geometry/24viz3h6,HR,https://gisco-services.ec.europa.eu/distributi...,10m,WGS84,"MULTIPOLYGON (((16.59681 46.4759, 16.85476 46...."
1480,http://data.europa.eu/nuts/geometry/6064mdm6,DE71,https://gisco-services.ec.europa.eu/distributi...,10m,WGS84,"POLYGON ((9.04243 50.49799, 9.10679 50.43799, ..."


### 1.3.3 Generate SPARQL update queries
For every column in the dataframe the function `get_geom_build_query` constructs a SPARQL-update query that contructs a `geo:Geometry` object containing triples compliant with the GeoSPARQL standards. These are then inserted in the graph.

Since GraphDB automatically recognises the GeoSPARQL data. GeoSPARQL geofunctions can now be used on the graph.

In [12]:
sparql_updates = df.apply(get_geom_build_query, axis=1)

In [13]:
# print(sparql_updates.iloc[0])

In [14]:
_ = sparql_updates.apply(sparql_update, update_endpoint_url)

C:\Users\arbeit\AppData\Local\Temp\ipykernel_13600\1421853152.py:1: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  _ = sparql_updates.apply(sparql_update, update_endpoint_url)


SUCESSFUL UPDATE
SUCESSFUL UPDATE
SUCESSFUL UPDATE
SUCESSFUL UPDATE
SUCESSFUL UPDATE
SUCESSFUL UPDATE
SUCESSFUL UPDATE
SUCESSFUL UPDATE
SUCESSFUL UPDATE
SUCESSFUL UPDATE


### 1.3.4 Test GeoSPARQL functionality

In [16]:
import folium
import geopandas as gpd
from shapely import wkt

In [17]:
with open("sparql/get_at22_neighbors.sparql", "r") as f:
    query_at22Neighbors = f.read()

at22_query = '''
PREFIX geo: <http://www.opengis.net/ont/geosparql#>

SELECT ?steiermark ?steiermark_geoWKT
WHERE {
  	?steiermark <http://purl.org/dc/elements/1.1/identifier> "AT22" .
    ?steiermark geo:hasGeometry ?steiermark_geo .
    ?steiermark_geo geo:asWKT ?steiermark_geoWKT .
}
'''

In [19]:
df

,which,all_WKT
0,http://data.europa.eu/nuts/code/SI03,"POLYGON ((16.59681 46.4759, 16.37983 46.54031,..."
1,http://data.europa.eu/nuts/code/AT21,"POLYGON ((13.35458 47.0973, 13.67774 47.03759,..."
2,http://data.europa.eu/nuts/code/AT11,"POLYGON ((17.06674 48.11868, 17.07721 48.03778..."
3,http://data.europa.eu/nuts/code/AT12,"POLYGON ((15.54245 48.9077, 15.75363 48.85218,..."
4,http://data.europa.eu/nuts/code/AT31,"POLYGON ((14.69101 48.5843, 14.89532 48.51252,..."
5,http://data.europa.eu/nuts/code/AT32,"MULTIPOLYGON (((13.30382 48.00782, 13.33794 47..."


In [20]:
df = sparql_select(query_at22Neighbors)
df['all_WKT'] = df['all_WKT'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry='all_WKT')

centroid = gdf.dissolve().geometry.centroid[0]
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=8)

df_steiermark = sparql_select(at22_query)
df_steiermark['steiermark_geoWKT'] = df_steiermark['steiermark_geoWKT'].apply(wkt.loads)
gdf_steiermark = gpd.GeoDataFrame(df_steiermark, geometry='steiermark_geoWKT')

geoms_json = gdf.geometry.to_json()
geoms_json_steiermark = gdf_steiermark.geometry.to_json()

neighbors = folium.GeoJson(data=geoms_json, style_function=lambda x: {"fillColor": "orange"})
gdf.apply(lambda x: folium.Popup("IRI:\n"+x['which']).add_to(neighbors), axis=1)
neighbors.add_to(m)

at22 = folium.GeoJson(data=geoms_json_steiermark, style_function=lambda x: {"fillColor": "blue"})
folium.Popup("IRI:\n"+gdf_steiermark['steiermark'][0]).add_to(at22)
at22.add_to(m)
m

# X Define geospatial tasks for KGs

## .1 Neighbors
Aggregating statistics with neighboring NUTS-regions:\
NUTS-areas are hierarchically organised. This means that aggregations of regions are predefined. For example the NUTS-region level 1 `AT1` encompasses the level 2 regions `AT11`, `AT12` and `AT13`. Statistics are collected on these regions. 

However, in some cases it could be interesting to analyse all bordering regions of a regions. For example the code `AT130`, level 3, (Vienna) neighbors `AT127`and `AT126` on the same level. These roughly encompass the Viennese metropolitain region, aggregating statistics for this case could be interesting, but would only be achievably with LLMs, if they clearly understand neighborhood relations.

Another potential use-case would be to look at each NUTS-region and calculate if it is richer or poorer than its neighbors.

It could also assist with calculation Moran's I for spatial autocorrelation.

## .2 Nearness
Nearness is not exactly defined. In this study the concept of nearness will be approximated by buffers. If we have a certain NUTS-region, it could be interesting if features are near them. For example the East-German region `DED43` is a rural area that battles with emigration. Taking a look at nearby cities (e.g. in a 50 km buffer zone) shows that Leipzig, Dresden an Chemnitz could be absobers of that emigration.

Answering such a question with an LLM would need it to reliably understand nearness in the form of a Buffer. 

## .3 Directions
What is east, west, north and south. Which NUTS-regions are west of region X. Which geographical features are north of region Y. Directions have to be clearly understood by the LLM in order to answer these questions.
Since directions are vague, assumptions and simplyfications will have to be made.

## .4 Between
Which features or other regions are between two NUTS-regions. What does between mean and can an LLM potentially capture this vagueness better that a clearly defined geographical approach.

In [ ]:
# ?OpenAI

In [ ]:
#ompletions

In [ ]:
codes = 

# Playground

In [ ]:
test_query = """
PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?codenum ?name ?wkt WHERE {
    ?code geo:hasGeometry ?geom .
    ?code <http://data.europa.eu/nuts/level> "1" .
    
    ?code <http://purl.org/dc/elements/1.1/identifier> ?codenum .
    ?code skos:altLabel ?name .
    
    ?geom geo:asWKT ?wkt .
}
"""
df = sparql_select(test_query)

df['wkt'] = df['wkt'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry='wkt')
gdf.to_file(r"C:\Users\arbeit\ucloud\New\00SOSE24\MASTERARBEIT\QGIS\qgis_data\nuts1.geojson")

In [35]:
df = sparql_select("""PREFIX geo: <http://www.opengis.net/ont/geosparql#>
PREFIX geof: <http://www.opengis.net/def/function/geosparql/>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>

SELECT ?which_code ?name  WHERE {
    ?code <http://purl.org/dc/elements/1.1/identifier> "DE14" .
    
    ?code geo:hasGeometry ?code_geo .
    ?code_geo geo:asWKT ?code_geoWKT .
    ?code <http://data.europa.eu/nuts/level> ?code_level .
    
    ?which_code <http://data.europa.eu/nuts/level> ?code_level .
    ?which_code geo:hasGeometry ?all_geos .
    ?all_geos geo:asWKT ?all_WKT .
    
    FILTER (geof:sfTouches(?code_geoWKT, ?all_WKT)) .

    ?which_code skos:prefLabel ?name .
}""", endpoint_url=select_endpoint_url)

In [36]:
df

,which_code,name
0,http://data.europa.eu/nuts/code/DE27,DE27 Schwaben
1,http://data.europa.eu/nuts/code/AT34,AT34 Vorarlberg
2,http://data.europa.eu/nuts/code/DE11,DE11 Stuttgart
3,http://data.europa.eu/nuts/code/DE12,DE12 Karlsruhe
4,http://data.europa.eu/nuts/code/DE13,DE13 Freiburg


## Get City Data from OSM

In [ ]:
import requests
import geopandas as gpd
import overpy

In [ ]:
####
nuts_dissolved = gpd.read_file("data_external/NUTSAREAS_mainland.geojson")
bbox = nuts_dissolved.bounds.values[0]
bbox

In [ ]:
query = f'''
node[place="city"]({bbox[1]},{bbox[0]},{bbox[3]},{bbox[2]});
(._;>;);
out body;
'''

In [ ]:
result = api.query(query)

In [ ]:
result.nodes

In [ ]:
result.nodes[0].

In [ ]:
result = api.query("""
    way(50.746,7.154,50.748,7.157) ["highway"];
    (._;>;);
    out body;
    """)